# 01 - Exploration visuelle & préparation des données
**Branche** : `feature/01-data-exploration`

**Objectifs** : vue d'ensemble du jeu, qualité (corruption, doublons, outliers), homogénéité des sources, nettoyage traçable.

> Hypothèse d'arborescence : `data/raw/labeled/{normal,cancer}/` et `data/raw/unlabeled/`.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from brainscanai.config import load_config, resolve_path
from brainscanai.utils import set_seed

cfg = load_config()
set_seed(cfg["seed"])
PROC, FEAT = resolve_path(cfg, "processed", create=True), resolve_path(cfg, "features", create=True)
FIG, TAB = resolve_path(cfg, "figures", create=True), resolve_path(cfg, "tables", create=True)
from brainscanai.data.loader import build_manifest
from brainscanai.data.quality import *
from brainscanai.evaluation.plots import plot_sample_grid, save_fig

## 1. Manifeste & structure

In [ ]:
manifest = build_manifest(cfg)
print(manifest.shape)
manifest.groupby(['source', 'label_name']).size().rename('n_images').to_frame()

## 2. Exemples d'images (par classe)

In [ ]:
rng = np.random.default_rng(cfg['seed'])
for name in ['normal', 'cancer', 'unlabeled']:
    sub = manifest[manifest.label_name == name]
    pick = sub.sample(min(5, len(sub)), random_state=cfg['seed'])
    fig = plot_sample_grid(pick.path.tolist(), [name] * len(pick))
    fig.suptitle(name, y=1.02); save_fig(fig, f'samples_{name}', FIG); plt.show()

## 3. Résolution, canaux, format, intensité, netteté
Le fichier de description annonce 512×512 JPEG ; l'énoncé de la mission parle de PNG : **on vérifie plutôt que de supposer**.

In [ ]:
stats = compute_image_stats(manifest)
print(stats[['width', 'height', 'mode', 'format']].value_counts(dropna=False))
stats[['mean', 'std', 'sharpness', 'black_ratio']].describe().round(2)

## 4. Outliers, doublons, fichiers corrompus
Z-score robuste (médiane/MAD). **Règle** : jamais de suppression d'une image *labellisée* (signalement seulement) ; doublons : on garde la copie labellisée.

In [ ]:
q = cfg['data']['quality']
stats = flag_duplicates(flag_outliers(stats, z=q['outlier_robust_z']).assign(md5=lambda d: d['md5'].fillna('x_' + d['image_id'].astype(str))))
print('corrompues :', int(stats.corrupt.sum()), '| doublons :', int(stats.is_duplicate.sum()), '| outliers :', int(stats.is_outlier.sum()))
out = stats[stats.is_outlier | stats.is_duplicate]
out.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, col in zip(axes, ['mean', 'std', 'sharpness']):
    for src, g in stats.groupby('source'):
        ax.hist(g[col].dropna(), bins=30, alpha=.6, density=True, label=src)
    ax.set_title(col); ax.legend()
save_fig(fig, 'intensity_by_source', FIG); plt.show()

## 5. Biais potentiels : labellisées vs non labellisées (test de Kolmogorov-Smirnov)
Un décalage de distribution (scanner, protocole) fausserait la propagation des labels.

In [ ]:
domain_shift_report(stats)

## 6. Nettoyage & sauvegarde

In [ ]:
cleaned, removed = clean_manifest(manifest, stats, q['drop_duplicates'], q['drop_unlabeled_outliers'])
print(f'{len(manifest)} -> {len(cleaned)} images ; retirées : {len(removed)}')
stats.to_csv(PROC / 'image_stats.csv', index=False)
cleaned.to_csv(PROC / 'manifest_clean.csv', index=False)
removed.to_csv(PROC / 'removed_images.csv', index=False)
removed.reason.value_counts()

## Observations
_À compléter après exécution sur le vrai jeu : résolution constante ? canaux ? outliers (artefacts, coupes quasi vides) ? écart labellisé / non labellisé ?_